<a href="https://colab.research.google.com/github/Jasp3r16/00_thesis_generative_timber/blob/main/24_25_optimizer_workflow_with_cost_and_ILP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Thesis: Reclaimed Timber in Deep Generative Design
**Notebook ID:** 24_25_optimizer_workflow_with_cost_and_ILP  
**Author:** Jasper Cluistra   
**Last Updated:** 2026-02-27
### Properties script
**Goal:** to generate a cost matrix for the geometry with use of the timber datasets, then using ILP to find the best matches   
**Inputs:**
*   CSV timber dataset
*   Digital geometry

**Outputs:**
*   Best match for each element in a structure

## Mounting Google drive

In [7]:
import os
import sys
from google.colab import drive

# 1. Mount Google Drive
drive.mount('/content/drive')

# 2. Definieer je paden (Pas de naam 'Thesis_Project' aan naar jouw mapnaam)
BASE_PATH = '/content/drive/MyDrive/Thesis_TU/'
DATA_PATH = os.path.join(BASE_PATH, 'data_thesis/')
DATA_IMPORT_PATH = os.path.join(BASE_PATH, 'data_import/')
SCRIPT_PATH = os.path.join(BASE_PATH, 'notebooks_thesis/')
OUTPUT_PATH = os.path.join(BASE_PATH, 'research_exports/')

# 3. Voeg de Scripts map toe aan het systeem-pad
# Hierdoor kun je .py bestanden uit die map 'importen'
if SCRIPT_PATH not in sys.path:
    sys.path.append(SCRIPT_PATH)

print(f"✅ Drive mounted. Project directory: {BASE_PATH}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Drive mounted. Project directory: /content/drive/MyDrive/Thesis_TU/


# IMPORTING

## Dataset

In [ ]:
import pandas as pd

# Laad de geëxporteerde dataset in je nieuwe omgeving
# Voeg sep=';' toe aan je inlaad-functie!
df_input_csv = pd.read_csv(DATA_PATH + 'complete_timber.csv', sep=';')

# Print de kolommen om zeker te weten dat ze goed gesplitst zijn
print("Gevonden kolommen:", df_input_csv.columns.tolist())

# Controleer of alle data, inclusief de RS_0000 ID's en binaire states, goed is overgekomen
print("\nDataset succesvol ingeladen. Hier zijn de eerste elementen:\n")
print(df_input_csv.head())

Gevonden kolommen: ['Member_ID', 'State', 'Length_Actual', 'Depth', 'Width', 'E_modulus_eff', 'f_mk', 'Density', 'Embodied Carbon Coëfficiënt', 'Transport_Dist', 'Emmisiefactor', 'Bewerkingsfactor']

Dataset succesvol ingeladen. Hier zijn de eerste elementen:

  Member_ID  State  Length_Actual  Depth  Width  E_modulus_eff  f_mk  Density  \
0  NS_00000      0         2400.0  100.0   38.0        11000.0    24      420   
1  NS_00001      0         2400.0  100.0   50.0        11000.0    24      420   
2  NS_00002      0         2400.0  100.0   75.0        11000.0    24      420   
3  NS_00003      0         2400.0  100.0  100.0        11000.0    24      420   
4  NS_00004      0         2400.0  150.0   38.0        11000.0    24      420   

   Embodied Carbon Coëfficiënt  Transport_Dist  Emmisiefactor  \
0                        150.0             500         0.1473   
1                        150.0             500         0.1019   
2                        150.0             500         0.

## Search space

De search space wordt vanuit het geometrie script geimporteerd, dan wordt een willekeurige samenstelling gekozen als beginpunt van de optimalisatie.


In [13]:
import json

json_path = DATA_IMPORT_PATH + 'search_space.json'
# Lees het "contract" in voor je optimizer
with open(json_path, 'r') as f:
    optimizer_search_space = json.load(f)

print(f"✅ Search Space ingeladen! De optimizer heeft {len(optimizer_search_space)} parameters om aan te draaien.")
# Nu kun je deze variabele direct aan je Optuna, PyTorch of Genetisch Algoritme voeren!

✅ Search Space ingeladen! De optimizer heeft 18 parameters om aan te draaien.


Dit script is bedoeld voor je Colab-omgeving. Het leest de search_space.json in, stelt de parameters dynamisch in op basis van jouw regels, en gebruikt een "dummy" voorspelling (waar later je echte Neurale Netwerk komt) om het beste ontwerp te vinden.

In [15]:
!pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 413.9/413.9 kB 7.2 MB/s eta 0:00:00


In [14]:
import json
import random
import numpy as np

# ==========================================
# 1. JSON INLADEN EN RANDOM SAMPLE GENEREREN
# ==========================================
def generate_random_design(json_path):
    """
    Leest de search space en genereert 1 willekeurige, geldige iteratie.
    """
    with open(json_path, 'r') as f:
        space = json.load(f)

    random_sample = {}

    for var_name, rules in space.items():
        if rules['type'] == 'discrete':
            # Kies een willekeurige stap uit de lijst
            random_sample[var_name] = random.choice(rules['options'])
        elif rules['type'] == 'continuous':
            # Kies een willekeurig decimaal getal tussen min en max
            random_sample[var_name] = random.uniform(rules['min'], rules['max'])

    return random_sample

# Genereer één ontwerp
huidig_ontwerp = generate_random_design(json_path)

print("1. Random Parametrisch Ontwerp Gegenereerd:")
# Print de eerste 5 parameters om te laten zien dat het werkt
for k, v in list(huidig_ontwerp.items())[:5]:
    print(f"   - {k}: {v:.3f}")


# ==========================================
# 2. VOXELIZATION (De Matrix van 0 en 1)
# ==========================================
print("\n2. Genereren van de 3D Binaire Matrix (Voxel Grid)...")

# Stel je constructie past maximaal in een doos van 6 bij 6 bij 3 meter.
# We verdelen de ruimte in "voxels" (kubusjes) van bijvoorbeeld 0.5 meter.
doos_x, doos_y, doos_z = 6.0, 6.0, 3.0
voxel_size = 0.5

# Bereken de afmetingen van de matrix
dim_x = int(doos_x / voxel_size) # 12 vakjes
dim_y = int(doos_y / voxel_size) # 12 vakjes
dim_z = int(doos_z / voxel_size) # 6 vakjes

# Maak een compleet lege 3D matrix aan gevuld met nullen
# Shape is (12, 12, 6)
voxel_matrix = np.zeros((dim_x, dim_y, dim_z), dtype=int)

# --- FICTIEVE PLAATSING (Om het concept te demonstreren) ---
# Normaal zou je Grasshopper of je eerdere Python geometrie-script hier
# de ECHTE coördinaten (x, y, z) van de knooppunten laten uitspugen.
# Voor nu nemen we even drie fictieve knooppunten die gegenereerd zijn:
knooppunten_xyz = [
    (0.0, 0.0, 0.0),    # Hoekpunt
    (2.625, 3.0, 0.0),  # v4_u en v4_v in werkelijke meters
    (4.1, 1.2, -1.875)  # Een verlaagd bottom punt
]

for (x, y, z) in knooppunten_xyz:
    # 1. Reken de (x, y, z) meters om naar matrix-indexen (0, 1, 2, 3...)
    # We pakken de absolute waarde voor de Z om negatieve dieptes om te zetten naar raster-indexen
    idx_x = int(x / voxel_size)
    idx_y = int(y / voxel_size)
    idx_z = int(abs(z) / voxel_size)

    # Zorg dat het punt niet buiten de doos valt (boundary check)
    idx_x = min(idx_x, dim_x - 1)
    idx_y = min(idx_y, dim_y - 1)
    idx_z = min(idx_z, dim_z - 1)

    # 2. Zet het vakje in de matrix op '1'
    voxel_matrix[idx_x, idx_y, idx_z] = 1

print(f"   ✅ 3D Matrix aangemaakt met afmetingen: {voxel_matrix.shape}")
print(f"   ✅ Aantal actieve voxels (1'en): {np.sum(voxel_matrix)}")

# Laat een klein 'plakje' van de bodem van de matrix zien (Z = 0)
print("\nBovenaanzicht van het raster op hoogte Z=0 (1 = knooppunt, 0 = leeg):")
print(voxel_matrix[:, :, 0])

1. Random Parametrisch Ontwerp Gegenereerd:
   - v1_shift_x: 0.375
   - v3_shift_y: -0.750
   - v4_shift_x: 0.375
   - v4_shift_y: -0.750
   - v5_shift_y: 0.000

2. Genereren van de 3D Binaire Matrix (Voxel Grid)...
   ✅ 3D Matrix aangemaakt met afmetingen: (12, 12, 6)
   ✅ Aantal actieve voxels (1'en): 3

Bovenaanzicht van het raster op hoogte Z=0 (1 = knooppunt, 0 = leeg):
[[1 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 1 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 0 0]]


# COST MATRIX

Omdat de pseudo-LCA berekening (uit de vorige stap) de impact van de gehele balk al berekent, heb je nu een methodologische keuze:

Optie A (De dubbele boete): Je telt de LCA-score én de penalty's bij elkaar op. Hiermee straf je afval extra zwaar af ten opzichte van het puur inbrengen van materiaal. Dit dwingt het algoritme agressief naar optimaal materiaalgebruik.

Optie B (De uitsplitsing): Je berekent de LCA-score alleen over het nuttige deel, en gebruikt je nieuwe formules om het verlies in kaart te brengen.

In [ ]:
# ==========================================
# CEL 1: INPUTS EN ALGEMENE PARAMETERS
# ==========================================
import pandas as pd
import numpy as np

# GWP Waarden (kg CO2 eq / kg hout)
GWP_VIRGIN = 0.50
GWP_RECLAIMED = 0.08

# LCA Parameters voor E_cost
PROCESSING_PENALTY_CO2 = 5.0  # kg CO2 boete voor bewerkingen (bijv. ontspijkeren)

print("✅ Parameters succesvol geladen.")

✅ Parameters succesvol geladen.


In [ ]:
# ==========================================
# CEL 2: DE REKENFUNCTIES (MODULES)
# ==========================================

def calculate_pseudo_lca_stock(df_stock):
    """
    Berekent de pseudo-LCA score (E_cost) voor een reeds ingeladen DataFrame
    met de timber stock.
    """
    # Maak een kopie om waarschuwingen (SettingWithCopyWarning) te voorkomen
    # en de originele input dataset zuiver te houden.
    df_lca = df_stock.copy()

    print("Start in-memory pseudo-LCA berekeningen...")

    # Stap A: Bereken Volume in m3 (dimensies zijn in mm, dus delen door 1000)
    df_lca['Volume_m3'] = (df_lca['Length_Actual'] / 1000) * (df_lca['Width'] / 1000) * (df_lca['Depth'] / 1000)

    # Stap B: Material Impact (A1-A3)
    df_lca['Impact_Material_kgCO2'] = df_lca['Volume_m3'] * df_lca['Embodied Carbon Coëfficiënt']

    # Stap C: Transport Impact (A4)
    # Dichtheid (Density) delen door 1000 om van kg/m3 naar ton/m3 te gaan
    df_lca['Impact_Transport_kgCO2'] = df_lca['Volume_m3'] * (df_lca['Density'] / 1000) * df_lca['Transport_Dist'] * df_lca['Emmisiefactor']

    # Stap D: Processing Impact (C3 / A3 Re-processing)
    # Binaire bewerkingsfactor (0 of 1) maal de vaste penalty
    df_lca['Impact_Processing_kgCO2'] = df_lca['Bewerkingsfactor'] * PROCESSING_PENALTY_CO2

    # Stap E: Totale E_cost Berekenen
    df_lca['E_cost_Total_kgCO2'] = (
        df_lca['Impact_Material_kgCO2'] +
        df_lca['Impact_Transport_kgCO2'] +
        df_lca['Impact_Processing_kgCO2']
    )

    return df_lca


def calculate_geometric_penalties(slot, stock_item):
    """
    Module 2: Berekent de CO2-penalty's voor zaagverlies en overdimensionering
    voor één specifieke match tussen een Slot (ontwerp) en Stock (voorraad).
    Geeft een tuple terug: (c_waste, c_overdim).
    """
    # 1. Haal afmetingen op en converteer naar meters
    l_slot = slot['Length_Req'] / 1000.0
    w_req = slot['Width_Req'] / 1000.0
    d_req = slot['Depth_Req'] / 1000.0
    a_req = w_req * d_req

    l_stock = stock_item['Length_Actual'] / 1000.0
    w_stock = stock_item['Width'] / 1000.0
    d_stock = stock_item['Depth'] / 1000.0
    a_stock = w_stock * d_stock

    rho = stock_item['Density'] # kg/m3

    # Bepaal GWP afhankelijk van de status (0=Nieuw, 1=Reclaimed)
    gwp_unit = GWP_RECLAIMED if stock_item['State'] == 1 else GWP_VIRGIN

    # 2. Bereken Zaagverlies (Lengteverlies) in kg CO2 eq
    c_waste = (l_stock - l_slot) * a_stock * rho * gwp_unit

    # 3. Bereken Overdimensionering (Dikteverlies) in kg CO2 eq
    c_overdim = (a_stock - a_req) * l_slot * rho * gwp_unit

    # Voorkom negatieve waardes (door afrondingsfouten of exact passende maten)
    c_waste = max(0, c_waste)
    c_overdim = max(0, c_overdim)

    return c_waste, c_overdim

print("✅ Rekenmodules succesvol gedefinieerd.")

✅ Rekenmodules succesvol gedefinieerd.


## Controle

In [ ]:
# 1. Voer de functie uit op je reeds ingeladen DataFrame
df_stock_with_lca = calculate_pseudo_lca_stock(df_input_csv)

# 2. Bekijk de resultaten van de berekening
print("\nPreview van de berekende E_cost per element:")
display(df_stock_with_lca.sample(10))

Start in-memory pseudo-LCA berekeningen...

Preview van de berekende E_cost per element:


,Member_ID,State,Length_Actual,Depth,Width,E_modulus_eff,f_mk,Density,Embodied Carbon Coëfficiënt,Transport_Dist,Emmisiefactor,Bewerkingsfactor,Volume_m3,Impact_Material_kgCO2,Impact_Transport_kgCO2,Impact_Processing_kgCO2,E_cost_Total_kgCO2
83,NS_00083,0,3000.0,300.0,100.0,11000.0,24,420,150.0,500,0.0840,0,0.090000,13.500000,1.587600,0.0,15.087600
88,NS_00088,0,3300.0,150.0,38.0,11000.0,24,420,150.0,500,0.1368,0,0.018810,2.821500,0.540374,0.0,3.361874
81,NS_00081,0,3000.0,300.0,50.0,11000.0,24,420,150.0,500,0.1425,0,0.045000,6.750000,1.346625,0.0,8.096625
177,RS_00010,1,5161.0,212.0,59.0,11000.0,24,420,15.0,88,0.0804,1,0.064554,0.968307,0.191827,5.0,6.160134
155,NS_00155,0,4000.0,200.0,100.0,11000.0,24,420,150.0,500,0.1429,0,0.080000,12.000000,2.400720,0.0,14.400720
141,NS_00141,0,4000.0,100.0,50.0,11000.0,24,420,150.0,500,0.1113,0,0.020000,3.000000,0.467460,0.0,3.467460
90,NS_00090,0,3300.0,150.0,75.0,11000.0,24,420,150.0,500,0.1365,0,0.037125,5.568750,1.064188,0.0,6.632938
148,NS_00148,0,4000.0,175.0,38.0,11000.0,24,420,150.0,500,0.1204,0,0.026600,3.990000,0.672554,0.0,4.662554
140,NS_00140,0,4000.0,100.0,38.0,11000.0,24,420,150.0,500,0.0993,0,0.015200,2.280000,0.316966,0.0,2.596966
169,RS_00002,1,11646.0,590.0,164.0,9000.0,18,380,15.0,32,0.0898,1,1.126867,16.903004,1.230503,5.0,23.133507




---



In [ ]:
# ==========================================
# CEL 3: HOOFDSCRIPT - BOUW DE COST MATRIX (GEFIXED)
# ==========================================

def build_optimization_matrix(df_design, df_stock_raw):
    print("Start generatie van de CO2 Cost Matrix...")

    # 1. Bereken de basis LCA score (Roep de module uit Cel 2 aan)
    df_stock = calculate_pseudo_lca_stock(df_stock_raw)

    n_slots = len(df_design)
    n_stock = len(df_stock)

    cost_matrix = np.full((n_slots, n_stock), np.inf)

    # Bijhouden hoeveel matches er daadwerkelijk lukken
    succesvolle_matches = 0

    for i in range(n_slots):
        slot = df_design.iloc[i]

        # Bepaal de grootste en kleinste doorsnedemaat van het SLOT (voor rotatie)
        slot_max_dim = max(slot['Width_Req'], slot['Depth_Req'])
        slot_min_dim = min(slot['Width_Req'], slot['Depth_Req'])

        for j in range(n_stock):
            stock_item = df_stock.iloc[j]

            # Bepaal de grootste en kleinste doorsnedemaat van de STOCK
            stock_max_dim = max(stock_item['Width'], stock_item['Depth'])
            stock_min_dim = min(stock_item['Width'], stock_item['Depth'])

            # --- HARD CONSTRAINTS (INCLUSIEF ROTATIE) ---
            if (stock_item['Length_Actual'] >= slot['Length_Req'] and
                stock_max_dim >= slot_max_dim and
                stock_min_dim >= slot_min_dim):

                # Het past! Haal de basis E_cost op
                e_cost_base = stock_item['E_cost_Base']

                # Roep Module 2 aan voor penalty's
                c_waste, c_overdim = calculate_geometric_penalties(slot, stock_item)

                # Totale score
                total_match_score = e_cost_base + c_waste + c_overdim
                cost_matrix[i, j] = total_match_score
                succesvolle_matches += 1

    print(f"✅ Matrix gegenereerd! Dimensies: {n_slots} x {n_stock}.")
    print(f"📊 Aantal fysiek geldige combinaties gevonden: {succesvolle_matches}")

    if succesvolle_matches == 0:
        print("\n⚠️ WAARSCHUWING: 0 matches gevonden. Je voorraad bevat geen elementen die groot genoeg zijn voor je ontwerp.")
        print(f"Grootste element in voorraad: Lengte={df_stock['Length_Actual'].max()}mm, Breedte={df_stock['Width'].max()}mm, Diepte={df_stock['Depth'].max()}mm")

    return cost_matrix, df_stock

# --- UITVOEREN VAN DE WORKFLOW ---

# Let op: Ik heb de test-inputs hier iets kleiner gemaakt,
# zodat ze zeker weten passen in de catalogus die we eerder hebben gegenereerd!
design_data = pd.DataFrame({
    'Element_ID': ['Kolom_1', 'Ligger_1', 'Ligger_2'],
    'Length_Req': [2000, 2500, 2500],  # mm (iets korter gemaakt)
    'Width_Req': [50, 75, 75],         # mm (past binnen je max 100mm TUPLE_WIDTH)
    'Depth_Req': [100, 150, 150]       # mm
})

cost_matrix, verrijkte_stock = build_optimization_matrix(design_data, df_input_csv)

# Print de matrix om te controleren of er nu getallen in staan in plaats van 'inf'
print("\nPreview Cost Matrix (eerste 3 rijen, eerste 5 kolommen):")
print(cost_matrix[:3, :5])

Start generatie van de CO2 Cost Matrix...
Start in-memory pseudo-LCA berekeningen...


KeyError: 'E_cost_Base'

# MATCHING ALGORITHM / ILP

## Input

## Script

In [ ]:
import pulp

# 1. DATA DEFINITIE
# De rijen (Stock / X) en kolommen (Constructie / Y)
stock_items = ['X1', 'X2', 'X3', 'X4N', 'X5N']
construction_slots = ['Y1', 'Y2', 'Y3', 'Y4', 'Y5', 'Y6']

# Jouw Cost Matrix (zoals je die in de tabel gaf)
# Let op: de volgorde moet overeenkomen met stock_items
cost_matrix = [
    [2,  6,  8,  3,  1,  4],   # X1
    [4,  5,  3,  6,  3,  2],   # X2
    [1,  7,  4,  9,  6,  5],   # X3
    [10, 10, 2,  2,  10, 10],  # X4 (Nieuw)
    [10, 1,  1,  10, 10, 10]   # X5 (Nieuw)
]

# We maken een dictionary om makkelijk kosten op te zoeken: costs['X1']['Y1'] = 2
costs = {
    stock_items[i]: {construction_slots[j]: cost_matrix[i][j]
                     for j in range(len(construction_slots))}
    for i in range(len(stock_items))
}

# 2. HET MODEL OPZETTEN
# We willen de kosten MINIMALISEREN
prob = pulp.LpProblem("Reclaimed_Timber_Matching", pulp.LpMinimize)

# 3. DECISION VARIABLES
# Dit maakt voor elke combinatie X-Y een variabele die 0 of 1 kan zijn (Binary)
# x['X1']['Y1'] wordt 1 als we matchen, anders 0.
x = pulp.LpVariable.dicts("Match", (stock_items, construction_slots), 0, 1, pulp.LpBinary)

# 4. OBJECTIVE FUNCTION (Doel)
# Som van alle (kosten * keuze)
prob += pulp.lpSum([x[i][j] * costs[i][j] for i in stock_items for j in construction_slots])

# 5. CONSTRAINTS (De Regels)
# Regel A: Elke Y (constructie onderdeel) MOET precies 1 stuk hout krijgen.
for j in construction_slots:
    prob += pulp.lpSum([x[i][j] for i in stock_items]) == 1

# Regel B: Capaciteit van de Stock (X)
# X1, X2, X3 zijn 'reclaimed' en uniek -> Maximaal 1x gebruiken
reclaimed_limit = 1
for i in ['X1', 'X2', 'X3']:
    prob += pulp.lpSum([x[i][j] for j in construction_slots]) <= reclaimed_limit

# X4, X5 zijn 'nieuw' -> Mogen vaker gebruikt worden (bijv. max 6x, want er zijn 6 gaten)
new_limit = 10000
for i in ['X4N', 'X5N']:
    prob += pulp.lpSum([x[i][j] for j in construction_slots]) <= new_limit

# 6. OPLOSSEN
prob.solve()

# 7. RESULTAAT PRINTEN
print(f"Status: {pulp.LpStatus[prob.status]}")
print("-" * 35)

total_cost = 0
print(f"{'Stock':<10} -> {'Element':<10} {'Kosten'}")

for j in construction_slots:
    for i in stock_items:
        if x[i][j].varValue == 1: # Als de beslissing 'JA' is
            print(f"{j:<10} -> {i:<10} {costs[i][j]}")
            total_cost += costs[i][j]

print("-" * 35)
print(f"Totale 'Penalty' Cost: {total_cost}")

Status: Optimal
-----------------------------------
Stock      -> Element    Kosten
Y1         -> X3         1
Y2         -> X5N        1
Y3         -> X5N        1
Y4         -> X4N        2
Y5         -> X1         1
Y6         -> X2         2
-----------------------------------
Totale 'Penalty' Cost: 8
